# Project 4 | NHS Data Engineering Portfolio | Yusuf Ismail
## NHS Cancer Waiting Times — Bronze Ingestion

**Notebook 01 of 5** — Bronze layer.

My Bronze layer has one job: take the four raw NHS England Combined CSVs exactly as they
are and write them to Parquet, adding only audit metadata so every row is traceable back
to its source. I do **not** clean, rename, filter, or transform anything here. That is
Silver's job. Bronze is my faithful, durable, columnar copy of the raw truth.

**Why this matters:** if I ever doubt a number in Silver or Gold, I can come back to Bronze
and see exactly what NHS England published, when I ingested it, and which file it came from.

**Input:** `data/raw/*.csv` (4 files — 3 full financial years plus 2025-26 Apr–Sep, Final only)
**Output:** `data/bronze/*.parquet` (one file per financial year)

**Audit columns I add to every row:**
- `_source_file` — the exact CSV filename the row came from
- `_financial_year` — the financial year label (2022-23, 2023-24, 2024-25, 2025-26-Apr-Sep)
- `_ingested_at` — UTC timestamp of when I ran this ingestion

**Principles I hold to in Bronze:**
- No cleaning, no renaming, no type coercion beyond what pandas does on read
- Nulls stay null (NHS suppresses small numbers — I never invent zeros)
- Idempotent — re-running this overwrites cleanly, same result every time

In [1]:
# I import pandas for the data, pathlib for portable paths, and datetime for my audit timestamp.
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

# I define my paths once. RAW_DIR is read-only in spirit; BRONZE_DIR is where I write.
PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "data" / "raw"
BRONZE_DIR = PROJECT_DIR / "data" / "bronze"

# I map each financial year to its raw file. This is my single source of truth for what to ingest.
# I map each financial year to its raw file. This is my single source of truth for what to ingest.
RAW_FILES = {
    "2022-23": RAW_DIR / "2022-23-Apr-Mar-Monthly-Combined-CSV-Final.csv",
    "2023-24": RAW_DIR / "2023-24-Apr-Mar-Monthly-Combined-CSV-Final.csv",
    "2024-25": RAW_DIR / "2024-25-Apr-Mar-Monthly-Combined-CSV-Final.csv",
    "2025-26-Apr-Sep": RAW_DIR / "2025-26-Apr-Sep-Cumulative-Monthly-Combined-CSV-Final.csv",
}

# I confirm my output folder exists before I try to write to it.
print("Bronze output folder exists:", BRONZE_DIR.exists())
print("Bronze output folder path:", BRONZE_DIR)

Bronze output folder exists: True
Bronze output folder path: /Users/yusufismail/Documents/cancer_waiting_time/data/bronze


## Step 1 — Define the ingestion function

I write one function that ingests a single file: read the CSV, stamp it with my three audit
columns, and return the dataframe. I keep it as a function (not a loop straight away) so I
can test it on one file, read the result with my own eyes, and only then run it across all three.

Note on `_ingested_at`: I capture the timestamp once, outside the function, so all three
files share the same ingestion run-time. That way I can tell at a glance which files were
ingested together in the same run.

In [3]:
# I capture one timestamp for the whole ingestion run, so every file from this run shares it.
INGESTION_TIME = datetime.now(timezone.utc).isoformat(timespec="seconds")
print("This ingestion run is stamped:", INGESTION_TIME)


def ingest_to_bronze(financial_year: str, csv_path: Path) -> pd.DataFrame:
    """I read one raw NHS CSV and add my audit columns. No cleaning, no transformation."""

    # I read the file exactly as NHS England published it. low_memory=False keeps column
    # types consistent across the wide, mostly-empty band columns.
    df = pd.read_csv(csv_path, low_memory=False)

    # I record the row count as-read, so I can prove later that nothing was lost in ingestion.
    rows_in = len(df)

    # I add my three audit columns. These are the only additions Bronze makes.
    df["_source_file"] = csv_path.name
    df["_financial_year"] = financial_year
    df["_ingested_at"] = INGESTION_TIME

    # I print a short receipt so I can see what happened to this file.
    print(f"  {financial_year}: read {rows_in:,} rows from {csv_path.name}")
    print(f"             added audit columns, now {df.shape[1]} columns total")

    return df


print("\nFunction defined. I will test it on one file next before running all three.")

This ingestion run is stamped: 2026-07-05T23:05:14+00:00

Function defined. I will test it on one file next before running all three.


## Step 2 — Test on one file before committing to all three

I run the function on the 2024-25 file only. I check three things:
1. The row count matches what I saw in exploration (324,758 rows)
2. My three audit columns are present and correctly populated
3. The original NHS data is untouched

If anything looks wrong here, I fix it before I ingest the other two years.

In [5]:
# I test the function on a single file first.
test_df = ingest_to_bronze("2024-25", RAW_FILES["2024-25"])

# I check the shape — I expect 324,758 rows and 33 columns (30 original + 3 audit).
print("\nShape after ingestion:", test_df.shape)

# I look at my three audit columns to confirm they populated correctly.
print("\nAudit columns (first 3 rows):")
print(test_df[["_source_file", "_financial_year", "_ingested_at"]].head(3).to_string(index=False))

# I confirm the original NHS columns are untouched by spot-checking the first row's key fields.
print("\nFirst row, original NHS fields (should be unchanged from raw):")
print(test_df[["Period", "Basis", "Standard_or_Item", "Cancer_Type", "Total", "Within"]].head(1).to_string(index=False))

  2024-25: read 324,758 rows from 2024-25-Apr-Mar-Monthly-Combined-CSV-Final.csv
             added audit columns, now 33 columns total

Shape after ingestion: (324758, 33)

Audit columns (first 3 rows):
                                  _source_file _financial_year              _ingested_at
2024-25-Apr-Mar-Monthly-Combined-CSV-Final.csv         2024-25 2026-07-05T23:05:14+00:00
2024-25-Apr-Mar-Monthly-Combined-CSV-Final.csv         2024-25 2026-07-05T23:05:14+00:00
2024-25-Apr-Mar-Monthly-Combined-CSV-Final.csv         2024-25 2026-07-05T23:05:14+00:00

First row, original NHS fields (should be unchanged from raw):
    Period    Basis Standard_or_Item Cancer_Type    Total   Within
2024-11-01 Provider              FDS ALL CANCERS 268572.0 207612.0


## Step 3 — Ingest all three years and write to Parquet

I loop over all three files, ingest each one, and write it to `data/bronze/` as Parquet.
I choose Parquet because it is columnar, compressed, preserves dtypes (including my nulls),
and loads far faster than CSV in Silver — exactly the format a real NHS data team would use.

I keep a small log of what I wrote so I have a receipt: row count, column count, and output
path for each year. I verify nothing was lost between read and write.

In [7]:
# I ingest and write each year in turn, keeping a log so I have a receipt of the whole run.
bronze_log = []

for fy, csv_path in RAW_FILES.items():
    # I ingest the file (read + stamp audit columns).
    df = ingest_to_bronze(fy, csv_path)

    # I write to Parquet. The filename mirrors the financial year for easy lookup.
    # index=False because the pandas row index carries no meaning I need to keep.
    out_path = BRONZE_DIR / f"cwt_bronze_{fy}.parquet"
    df.to_parquet(out_path, index=False)

    # I record the receipt for this file.
    bronze_log.append({
        "financial_year": fy,
        "rows": len(df),
        "columns": df.shape[1],
        "output_file": out_path.name,
        "size_mb": round(out_path.stat().st_size / 1_000_000, 1),
    })
    print(f"             wrote {out_path.name} ({bronze_log[-1]['size_mb']} MB)\n")

# I show the full ingestion log as a tidy table.
print("=== Bronze ingestion log ===")
print(pd.DataFrame(bronze_log).to_string(index=False))

  2022-23: read 285,645 rows from 2022-23-Apr-Mar-Monthly-Combined-CSV-Final.csv
             added audit columns, now 33 columns total
             wrote cwt_bronze_2022-23.parquet (4.9 MB)

  2023-24: read 304,027 rows from 2023-24-Apr-Mar-Monthly-Combined-CSV-Final.csv
             added audit columns, now 33 columns total
             wrote cwt_bronze_2023-24.parquet (5.2 MB)

  2024-25: read 324,758 rows from 2024-25-Apr-Mar-Monthly-Combined-CSV-Final.csv
             added audit columns, now 33 columns total
             wrote cwt_bronze_2024-25.parquet (5.5 MB)

  2025-26-Apr-Sep: read 162,435 rows from 2025-26-Apr-Sep-Cumulative-Monthly-Combined-CSV-Final.csv
             added audit columns, now 33 columns total
             wrote cwt_bronze_2025-26-Apr-Sep.parquet (2.7 MB)

=== Bronze ingestion log ===
 financial_year   rows  columns                        output_file  size_mb
        2022-23 285645       33         cwt_bronze_2022-23.parquet      4.9
        2023-24 304027  

## Step 4 — Read-back verification

Writing the files is not enough. I reopen each Parquet file and confirm:
1. The row counts match exactly what I wrote
2. My audit columns survived the round-trip
3. Nulls are still nulls (Parquet preserves them — I check a suppressed value to be sure)

Only once this passes do I consider Bronze complete and trustworthy.

In [9]:
# I read every Parquet file back and verify it matches what I wrote.
print("=== Read-back verification ===\n")

total_rows = 0
for fy in RAW_FILES.keys():
    path = BRONZE_DIR / f"cwt_bronze_{fy}.parquet"
    df = pd.read_parquet(path)
    total_rows += len(df)

    # I confirm row count, audit columns present, and the financial year stamp is correct.
    audit_ok = all(c in df.columns for c in ["_source_file", "_financial_year", "_ingested_at"])
    fy_ok = (df["_financial_year"] == fy).all()

    print(f"{fy}: {len(df):,} rows | audit columns present: {audit_ok} | FY stamp correct: {fy_ok}")

print(f"\nTotal rows across all three Bronze files: {total_rows:,}")

# I prove nulls survived by checking one suppressed-style column.
# In exploration I saw 'After' is null for the referral items — I confirm that holds after round-trip.
sample = pd.read_parquet(BRONZE_DIR / "cwt_bronze_2024-25.parquet")
usc_after_nulls = sample[sample["Standard_or_Item"] == "Urgent Suspected Cancer referral"]["After"].isna().all()
print(f"Nulls preserved (USC 'After' all null, as in raw): {usc_after_nulls}")

=== Read-back verification ===

2022-23: 285,645 rows | audit columns present: True | FY stamp correct: True
2023-24: 304,027 rows | audit columns present: True | FY stamp correct: True
2024-25: 324,758 rows | audit columns present: True | FY stamp correct: True
2025-26-Apr-Sep: 162,435 rows | audit columns present: True | FY stamp correct: True

Total rows across all three Bronze files: 1,076,865
Nulls preserved (USC 'After' all null, as in raw): True
